In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 7 — UMABC (Utility Mining-Guided ABC, proposé)
# Eq.(1) HUFSM · Eq.(2) Anchor-Shift · Eq.(3) Memory-Driven Scout
# ═══════════════════════════════════════════════════════════════════

class UMABCOptimizer:
    """
    Utility Mining-Guided Artificial Bee Colony (UMABC).

    Eq.(1)  U(xi) = α·fit_norm(xi) + (1-α)·dist_norm(xi)
    Eq.(2)  v_{i,j} = x_HUFS,j + φ·(x_{i,j} - x_{k,j})
    Eq.(3)  V_{i,j} = r1·x_bestHUFS + r2·x_rand + r3·(L+r4·(U-L))
    """
    def __init__(self, fid, SN, MaxFEs, limit, D, alpha=0.6, hufs_ratio=0.2):
        self.fid = fid
        self.SN, self.MaxFEs, self.limit, self.D = SN, MaxFEs, limit, D
        self.alpha = alpha
        self.hufs_ratio = hufs_ratio
        self.lb, self.ub = get_bounds(fid)

    def _utility(self, pop, fvals):
        """Eq.(1) : U(xi) = α·fit_norm + (1-α)·dist_norm"""
        fa = np.array([fitness_val(f) for f in fvals])
        fa_range = fa.max() - fa.min()
        fn = (fa - fa.min()) / (fa_range + 1e-10)

        d = np.zeros(self.SN)
        for i in range(self.SN):
            di = np.linalg.norm(pop - pop[i], axis=1)
            di[i] = np.inf
            d[i] = di.min()
        d_range = d.max() - d.min()
        dn = (d - d.min()) / (d_range + 1e-10)

        return self.alpha * fn + (1 - self.alpha) * dn

    @staticmethod
    def _safe_probs(utils):
        """
        Convertit un vecteur d'utilités en probabilités valides pour
        np.random.choice. Robuste à tous les cas dégénérés :
          - tous identiques (utils = 0) → distribution uniforme
          - valeurs très petites (underflow) → distribution uniforme
          - NaN / Inf → distribution uniforme
        La somme est forcée à 1.0 exactement pour satisfaire numpy.
        """
        s = np.sum(utils)
        if not np.isfinite(s) or s < 1e-10:
            # Distribution uniforme : fallback sûr quand utils dégénéré
            n = len(utils)
            return np.full(n, 1.0 / n)
        pr = utils / s
        # Renormaliser pour garantir sum == 1.0 au bit près (float64)
        pr = pr / pr.sum()
        return pr

    def _hufs(self, pop, fvals):
        """Retourne les k individus de plus haute utilité (HUFS set)."""
        utils = self._utility(pop, fvals)
        k     = max(1, int(self.SN * self.hufs_ratio))
        idx   = np.argsort(utils)[::-1][:k]
        return pop[idx], fvals[idx]

    def run(self, seed=None):
        if seed is not None:
            np.random.seed(seed)
        lb, ub, D, SN = self.lb, self.ub, self.D, self.SN

        pop   = rand_pop(SN, D, lb, ub)
        fvals = np.array([evaluate(self.fid, x) for x in pop])
        trial = np.zeros(SN, int)
        best_val = fvals.min()
        NFE = SN
        history = [best_val]

        while NFE < self.MaxFEs:
            # ── HUFSM — Eq.(1) ─────────────────────────────────────────
            hufs, hufs_fv = self._hufs(pop, fvals)
            x_best = hufs[np.argmin(hufs_fv)]

            # ── Employed — Eq.(2) Anchor-Shift ─────────────────────────
            for i in range(SN):
                if NFE >= self.MaxFEs: break
                anc = hufs[np.random.randint(len(hufs))]
                k   = np.random.choice([j for j in range(SN) if j != i])
                jj  = np.random.randint(D)
                v   = pop[i].copy()
                v[jj] = np.clip(
                    anc[jj] + np.random.uniform(-1, 1) * (pop[i, jj] - pop[k, jj]),
                    lb, ub)
                fv = evaluate(self.fid, v); NFE += 1
                if fv <= fvals[i]:
                    pop[i] = v; fvals[i] = fv; trial[i] = 0
                else:
                    trial[i] += 1
                if fv < best_val:
                    best_val = fv

            # ── Onlooker — Utility probs + Eq.(2) ──────────────────────
            utils = self._utility(pop, fvals)
            pr    = self._safe_probs(utils)          # ← robuste, jamais ValueError
            for _ in range(SN):
                if NFE >= self.MaxFEs: break
                i   = np.random.choice(SN, p=pr)
                anc = hufs[np.random.randint(len(hufs))]
                k   = np.random.choice([j for j in range(SN) if j != i])
                jj  = np.random.randint(D)
                v   = pop[i].copy()
                v[jj] = np.clip(
                    anc[jj] + np.random.uniform(-1, 1) * (pop[i, jj] - pop[k, jj]),
                    lb, ub)
                fv = evaluate(self.fid, v); NFE += 1
                if fv <= fvals[i]:
                    pop[i] = v; fvals[i] = fv; trial[i] = 0
                else:
                    trial[i] += 1
                if fv < best_val:
                    best_val = fv

            # ── Scout — Eq.(3) Memory-Driven Restart ───────────────────
            for i in range(SN):
                if NFE >= self.MaxFEs: break
                if trial[i] > self.limit:
                    r1, r2, r3, r4 = np.random.uniform(0, 1, 4)
                    xr = pop[np.random.randint(SN)]
                    pop[i] = np.clip(
                        r1 * x_best + r2 * xr + r3 * (lb + r4 * (ub - lb)),
                        lb, ub)
                    fvals[i] = evaluate(self.fid, pop[i]); NFE += 1
                    trial[i] = 0
                    if fvals[i] < best_val:
                        best_val = fvals[i]

            history.append(best_val)

        return best_val, np.array(history)

print('✓ UMABCOptimizer prêt — _safe_probs() corrige ValueError probabilités.')
